# 🌿 CropGuard: A Data Mining Approach for Crop Disease Detection
### PlantVillage Dataset — Exploratory Data Analysis
---
**Project Type:** Data Mining Project

**Dataset:** PlantVillage (54,305 images — 38 disease classes)  
**Source:** Kaggle — emmarex/plantdisease

## Step 1 — Import Libraries

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial.distance import euclidean
import kagglehub

print("All libraries imported successfully!")

All libraries imported successfully!


C:\Users\nadaa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 — Download & Load the Dataset
Downloads PlantVillage automatically from Kaggle, then reads all images and extracts color features (R, G, B channels + brightness) into a DataFrame.

In [ ]:
# Download PlantVillage dataset automatically from Kaggle
path = kagglehub.dataset_download("emmarex/plantdisease")
print("Path to dataset files:", path)

# The dataset folder is inside the downloaded path
DATASET_PATH = os.path.join(path, "PlantVillage")

# Walk through all folders and collect image info
rows = []

for class_folder in os.listdir(DATASET_PATH):
    class_path = os.path.join(DATASET_PATH, class_folder)

    if os.path.isdir(class_path):
        images = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.JPG', '.png'))]

        for img_file in images:
            img_path = os.path.join(class_path, img_file)
            try:
                img = Image.open(img_path).convert('RGB')
                img_resized = img.resize((64, 64))          # resize to 64x64
                arr = np.array(img_resized).astype(float)   # convert to array

                rows.append({
                    "filename":   img_file,
                    "class":      class_folder,
                    "mean_R":     arr[:, :, 0].mean(),      # average Red channel
                    "mean_G":     arr[:, :, 1].mean(),      # average Green channel
                    "mean_B":     arr[:, :, 2].mean(),      # average Blue channel
                    "std_R":      arr[:, :, 0].std(),
                    "std_G":      arr[:, :, 1].std(),
                    "std_B":      arr[:, :, 2].std(),
                    "brightness": arr.mean(),               # overall brightness
                })
            except Exception:
                pass   # skip corrupted images

df = pd.DataFrame(rows)
print("Dataset loaded successfully!")
print(f"Total images: {len(df)}")

## Step 3 — Basic Statistics


In [ ]:
print("=== Dataset Shape ===")
print(df.shape)
print()

In [ ]:
print("=== Basic Statistics ===")
df.describe()

In [ ]:
print("=== First 20 Rows ===")
df.head(20)

## Step 4 — Detect Missing Values

In [ ]:
print("=== Missing Values Per Column ===")
print(df.isnull().sum())

In [ ]:
print("=== Missing Values Map (True = missing) ===")
df.isnull()

## Step 5 — Handle Missing Values


In [ ]:
df_clean = df.dropna()

# Fill remaining missing values with median (safer for image features)
df = df.fillna(df.median(numeric_only=True))

print("=== After Handling Missing Values ===")
print(df.isnull().sum())

## Step 6 — Class Distribution
How many images exist per disease class — like checking column values in the diabetes example.

In [ ]:
print("=== Number of Images Per Disease Class ===")
class_counts = df["class"].value_counts()
print(class_counts)

In [ ]:
# Check if any class has less than 10 images
print("=== Classes with Less Than 10 Images ===")
print(class_counts[class_counts < 10])

## Step 9 — Euclidean Distance


In [ ]:
features = ["mean_R", "mean_G", "mean_B", "brightness"]

row1 = df[features].iloc[0].astype(float).values
row2 = df[features].iloc[1].astype(float).values

dist = euclidean(row1, row2)
print(f"=== Euclidean Distance Between Image 1 and Image 2 ===")
print(f"Distance: {dist:.4f}")

## Step 10 — NumPy Operations


In [ ]:
brightness_array = np.array(df["brightness"])

print("=== NumPy Operations on Brightness Column ===")
print(f"Mean brightness:  {np.mean(brightness_array):.2f}")
print(f"Std  brightness:  {np.std(brightness_array):.2f}")
print(f"Min  brightness:  {np.min(brightness_array):.2f}")
print(f"Max  brightness:  {np.max(brightness_array):.2f}")
print()
print("=== Any NaN in brightness array? ===")
print(np.isnan(brightness_array).sum(), "NaN values found")

## Step 11 — Visualization


In [ ]:
plt.figure(figsize=(14, 6))
class_counts.plot(kind='bar', color='green', edgecolor='black')
plt.title("Number of Images per Disease Class - PlantVillage")
plt.xlabel("Disease Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("class_distribution.png")
plt.show()
print("Chart saved as class_distribution.png")